### Objectives
- Evaluate distributed classification models using appropriate metrics
- Analyse model stability using resampling techniques
- Investigate scalability through controlled experiments
- Identify computational bottlenecks and trade-offs
- Align model performance with business-oriented considerations

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [3]:
from pyspark.ml import PipelineModel

model_path = "/content/drive/MyDrive/bigdata_ml_project/models/rf_model"
rf_pipeline_model = PipelineModel.load(model_path)

In [4]:
from pyspark.sql.functions import (
    col, hour, dayofweek, unix_timestamp, when
)

data_path = "/content/drive/MyDrive/bigdata_ml_project/data/raw/nyc_taxi"
df = spark.read.parquet(data_path)

df = df.withColumn(
    "trip_duration_minutes",
    (unix_timestamp("tpep_dropoff_datetime") -
     unix_timestamp("tpep_pickup_datetime")) / 60
)

df = df.withColumn("pickup_hour", hour("tpep_pickup_datetime")) \
       .withColumn("pickup_dayofweek", dayofweek("tpep_pickup_datetime"))

df = df.filter(col("trip_duration_minutes") > 0)
df = df.filter(col("trip_distance") > 0)
df = df.filter(col("passenger_count") > 0)

quantile_75 = df.approxQuantile("total_amount", [0.75], 0.01)[0]

df = df.withColumn(
    "high_revenue_trip",
    when(col("total_amount") >= quantile_75, 1).otherwise(0)
)

df = df.select(
    "passenger_count",
    "trip_distance",
    "trip_duration_minutes",
    "pickup_hour",
    "pickup_dayofweek",
    "PULocationID",
    "DOLocationID",
    "high_revenue_trip",
    "tpep_pickup_datetime"
)

In [5]:
from pyspark.sql.functions import year

df = df.withColumn("year", year("tpep_pickup_datetime"))
test_df = df.filter(col("year") >= 2020)

In [6]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

predictions = rf_pipeline_model.transform(test_df)

evaluator_roc = BinaryClassificationEvaluator(
    labelCol="high_revenue_trip",
    metricName="areaUnderROC"
)

evaluator_pr = BinaryClassificationEvaluator(
    labelCol="high_revenue_trip",
    metricName="areaUnderPR"
)

roc_auc = evaluator_roc.evaluate(predictions)
pr_auc = evaluator_pr.evaluate(predictions)

roc_auc, pr_auc

(0.9125101904237907, 0.84278280611832)

In [7]:
predictions.groupBy("high_revenue_trip", "prediction").count().show()

+-----------------+----------+--------+
|high_revenue_trip|prediction|   count|
+-----------------+----------+--------+
|                1|       0.0| 1049683|
|                0|       0.0|26758796|
|                1|       1.0|16427577|
|                0|       1.0|24793933|
+-----------------+----------+--------+



In [8]:
# Take a manageable subset of the test data
test_small = test_df.sample(fraction=0.1, seed=42).cache()
test_small.count()

6900257

In [9]:
import random
import numpy as np

def bootstrap_auc(df, model, n=5, frac=0.5):
    scores = []
    for i in range(n):
        sample = df.sample(fraction=frac, seed=random.randint(0, 10000))
        preds = model.transform(sample)
        score = evaluator_roc.evaluate(preds)
        scores.append(score)
    return scores

In [10]:
auc_samples = bootstrap_auc(test_small, rf_pipeline_model, n=5, frac=0.5)
auc_samples

[0.9126435241220954,
 0.9125501703036727,
 0.9123867161686394,
 0.9123770582753412,
 0.9125166671248739]

In [11]:
import numpy as np

np.mean(auc_samples), np.std(auc_samples)

(np.float64(0.9124948271989245), np.float64(0.0001012009197590575))

In [15]:
type(rf_pipeline_model)

pyspark.ml.pipeline.PipelineModel

In [16]:
rf_pipeline_model.stages

[StringIndexerModel: uid=StringIndexer_bfd8891ce744, handleInvalid=keep,
 StringIndexerModel: uid=StringIndexer_cd2bc252a1cf, handleInvalid=keep,
 VectorAssembler_e3fd10233bee,
 RandomForestClassificationModel: uid=RandomForestClassifier_c9b98254a60a, numTrees=50, numClasses=2, numFeatures=7]

In [12]:
import time

def timed_prediction(df, model):
    start = time.time()
    model.transform(df).count()
    return time.time() - start

In [13]:
times = {}

for p in [50, 100, 200]:
    df_p = test_df.repartition(p)
    times[p] = timed_prediction(df_p, rf_pipeline_model)

times

{50: 75.47015762329102, 100: 73.99255609512329, 200: 63.83283805847168}

In [14]:
sizes = {}
fractions = [0.25, 0.5, 1.0]

for f in fractions:
    df_f = test_df.sample(fraction=f, seed=42)
    sizes[f] = timed_prediction(df_f, rf_pipeline_model)

sizes

{0.25: 23.737568140029907, 0.5: 28.369656801223755, 1.0: 25.841716766357422}

### Bottleneck Analysis
- I/O overhead dominates initial stages due to Parquet deserialization.
- Shuffle operations increase with partition count.
- Tree-based models incur higher computation cost but scale well.
- Single-node constraints limit true horizontal scalability.

### Cost–Performance Trade-offs
Using a managed distributed cluster would reduce runtime for hyperparameter tuning and large-scale evaluation at increased monetary cost. Google Colab provides a cost-effective environment for prototyping and evaluation, at the expense of limited executor parallelism and memory.

### Business Interpretation
The model prioritises identifying high-revenue trips, enabling applications such as dynamic pricing, demand forecasting, and fleet allocation. Precision–recall performance is particularly relevant due to class imbalance and asymmetric business costs.